# Baseline Massey Notebook

**Simple, fast baseline model** using:
- Win percentage (home/away/neutral)
- Season scoring stats
- Seed differential
- Massey Ordinal rankings (men's only — not available for women's)

XGBoost classifier + isotonic spline on seed diff for calibration.

**Use this as a sanity-check baseline.** It runs in ~2 minutes vs ~10 for elo_enhanced.

Output: `output/{CURRENT_SEASON}/baseline_massey/submission_combined.csv`

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================
CURRENT_SEASON = 2026
# DATA_DIR options:
#   "../data/{CURRENT_SEASON}"  — year-specific dir (men's only for some years)
#   "../data/2026"              — cumulative through 2025 (has both M and W; use for backtesting)
DATA_DIR = f"../data/{CURRENT_SEASON}"
BLEND_WEIGHT_MASSEY = 0.7  # weight on spline vs model (0=all model, 1=all spline)
# =============================================================================

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
from scipy.interpolate import UnivariateSpline

sys.path.insert(0, os.path.abspath('..'))

def make_output_path(year, method, gender, filename):
    path = f"../output/{year}/{method}/{gender}"
    os.makedirs(path, exist_ok=True)
    return f"{path}/{filename}"

os.makedirs("../output", exist_ok=True)
print(f"Season: {CURRENT_SEASON}")

## Feature Engineering

In [ ]:
def compute_win_pcts(results_df, season_filter=None):
    """Compute home/away/neutral win percentages per team-season."""
    if season_filter:
        results_df = results_df[results_df['Season'].isin(season_filter)]

    rows = []
    for season, grp in results_df.groupby('Season'):
        # wins
        for loc, mask in [('H', grp['WLoc']=='H'), ('A', grp['WLoc']=='A'), ('N', grp['WLoc']=='N')]:
            w = grp[mask].groupby('WTeamID').size().rename('wins')
            l_loc = {'H':'A','A':'H','N':'N'}[loc]
            l_mask = grp['WLoc'] == l_loc
            l = grp[l_mask].groupby('LTeamID').size().rename('losses')
            combined = pd.concat([w, l], axis=1).fillna(0)
            combined['total'] = combined['wins'] + combined['losses']
            combined['pct'] = combined['wins'] / combined['total'].clip(lower=1)
            combined['Season'] = season
            combined['loc'] = loc
            rows.append(combined.reset_index().rename(columns={'WTeamID':'TeamID'}))

        # Overall win pct
        wins = grp.groupby('WTeamID').size()
        losses = grp.groupby('LTeamID').size()
        idx = wins.index.union(losses.index)
        w_all = wins.reindex(idx, fill_value=0)
        l_all = losses.reindex(idx, fill_value=0)
        overall = pd.DataFrame({'wins': w_all, 'losses': l_all})
        overall['total'] = overall['wins'] + overall['losses']
        overall['pct'] = overall['wins'] / overall['total'].clip(lower=1)
        overall['Season'] = season
        overall['loc'] = 'Overall'
        rows.append(overall.reset_index().rename(columns={'index':'TeamID'}))

    df = pd.concat(rows, ignore_index=True)
    return df.pivot_table(index=['Season','TeamID'], columns='loc', values='pct').reset_index()


In [ ]:
def compute_massey_features(massey_df, cutoff_day=128):
    """Extract normalized Massey ordinal rankings at tournament cutoff."""
    late = massey_df[massey_df['RankingDayNum'] <= cutoff_day].copy()
    # Use most recent ranking for each team-season across all systems
    latest = (
        late.sort_values('RankingDayNum', ascending=False)
        .groupby(['Season','TeamID'], as_index=False)
        .first()
    )
    # Average rank across systems per team-season
    avg_rank = (
        late.groupby(['Season','TeamID'])['OrdinalRank'].mean()
        .reset_index(name='MasseyRankAvg')
    )
    # Normalize within season (lower rank = better team, so invert)
    avg_rank['MasseyRankNorm'] = avg_rank.groupby('Season')['MasseyRankAvg'].transform(
        lambda x: 1 - (x - x.min()) / (x.max() - x.min() + 1e-9)
    )
    return avg_rank[['Season','TeamID','MasseyRankAvg','MasseyRankNorm']]

In [ ]:
def build_matchup_features(tourney_df, win_pct_df, seeds_df, massey_df=None):
    """Build feature matrix for historical tournament matchups."""
    rows = []
    for _, g in tourney_df.iterrows():
        s = g['Season']
        w, l = g['WTeamID'], g['LTeamID']
        t1, t2 = min(w,l), max(w,l)
        actual = 1.0 if t1 == w else 0.0

        def get_wp(tid, col):
            row = win_pct_df[(win_pct_df['Season']==s) & (win_pct_df['TeamID']==tid)]
            return row[col].values[0] if len(row) and col in row.columns else 0.5

        def get_seed(tid):
            row = seeds_df[(seeds_df['Season']==s) & (seeds_df['TeamID']==tid)]
            if len(row): return int(row.iloc[0]['Seed'][1:3])
            return 8

        seed1, seed2 = get_seed(t1), get_seed(t2)
        feat = {
            'Season': s, 'Team1ID': t1, 'Team2ID': t2,
            'SeedDiff': seed1 - seed2,
            'WPct1': get_wp(t1, 'Overall'), 'WPct2': get_wp(t2, 'Overall'),
            'WPctH1': get_wp(t1, 'H'), 'WPctH2': get_wp(t2, 'H'),
            'WPctA1': get_wp(t1, 'A'), 'WPctA2': get_wp(t2, 'A'),
            'WPctN1': get_wp(t1, 'N'), 'WPctN2': get_wp(t2, 'N'),
            'Result': actual,
        }

        if massey_df is not None:
            def get_massey(tid, col):
                row = massey_df[(massey_df['Season']==s) & (massey_df['TeamID']==tid)]
                return row[col].values[0] if len(row) else 0.5
            feat['Massey1'] = get_massey(t1, 'MasseyRankNorm')
            feat['Massey2'] = get_massey(t2, 'MasseyRankNorm')
            feat['MasseyDiff'] = feat['Massey1'] - feat['Massey2']

        rows.append(feat)

    return pd.DataFrame(rows)

## Men's Model

In [ ]:
reg_m = pd.read_csv(f"{DATA_DIR}/MRegularSeasonCompactResults.csv")
tourney_m = pd.read_csv(f"{DATA_DIR}/MNCAATourneyCompactResults.csv")
seeds_m = pd.read_csv(f"{DATA_DIR}/MNCAATourneySeeds.csv")
massey_m = pd.read_csv(f"{DATA_DIR}/MMasseyOrdinals.csv")
teams_m = pd.read_csv(f"{DATA_DIR}/MTeams.csv")

win_pct_m = compute_win_pcts(reg_m)
massey_feats_m = compute_massey_features(massey_m)

train_tourney_m = tourney_m[(tourney_m['Season'] >= 2010) & (tourney_m['Season'] < CURRENT_SEASON)]
features_m = build_matchup_features(train_tourney_m, win_pct_m, seeds_m, massey_feats_m)

feat_cols = [c for c in features_m.columns if c not in ['Season','Team1ID','Team2ID','Result']]
X_m = features_m[feat_cols].fillna(0)
y_m = features_m['Result']

print(f"Training samples: {len(X_m)}, features: {feat_cols}")

In [ ]:
model_m = xgb.XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    use_label_encoder=False, eval_metric='logloss',
    random_state=42, verbosity=0,
)
model_m.fit(X_m, y_m)
print("Men's XGBoost trained.")

In [ ]:
# Isotonic spline on seed diff for calibration
seed_diff_vals = features_m['SeedDiff'].values
seed_spline = UnivariateSpline(
    sorted(seed_diff_vals),
    features_m.sort_values('SeedDiff')['Result'].values,
    s=len(seed_diff_vals), ext=3,
)

def predict_matchup(t1, t2, season, model, win_pct_df, massey_df, seeds_df, spline, blend=0.3):
    feat = {
        'SeedDiff': 0, 'WPct1':0.5,'WPct2':0.5,'WPctH1':0.5,'WPctH2':0.5,
        'WPctA1':0.5,'WPctA2':0.5,'WPctN1':0.5,'WPctN2':0.5,
        'Massey1':0.5,'Massey2':0.5,'MasseyDiff':0,
    }
    def gw(tid, col):
        r = win_pct_df[(win_pct_df['Season']==season)&(win_pct_df['TeamID']==tid)]
        return r[col].values[0] if len(r) and col in r.columns else 0.5
    def gs(tid):
        r = seeds_df[(seeds_df['Season']==season)&(seeds_df['TeamID']==tid)]
        if len(r): return int(r.iloc[0]['Seed'][1:3])
        return 8
    def gm(tid, col):
        r = massey_df[(massey_df['Season']==season)&(massey_df['TeamID']==tid)]
        return r[col].values[0] if len(r) else 0.5

    feat['WPct1'] = gw(t1,'Overall'); feat['WPct2'] = gw(t2,'Overall')
    feat['WPctH1'] = gw(t1,'H'); feat['WPctH2'] = gw(t2,'H')
    feat['WPctA1'] = gw(t1,'A'); feat['WPctA2'] = gw(t2,'A')
    feat['WPctN1'] = gw(t1,'N'); feat['WPctN2'] = gw(t2,'N')
    feat['SeedDiff'] = gs(t1) - gs(t2)
    feat['Massey1'] = gm(t1,'MasseyRankNorm'); feat['Massey2'] = gm(t2,'MasseyRankNorm')
    feat['MasseyDiff'] = feat['Massey1'] - feat['Massey2']

    row = pd.DataFrame([feat])[feat_cols].fillna(0)
    model_pred = float(model.predict_proba(row)[0, 1])
    # spline(SeedDiff): SeedDiff<0 means lo has better seed -> higher win prob for lo
    spline_pred = float(np.clip(spline(feat['SeedDiff']), 0.025, 0.975))
    return blend * model_pred + (1 - blend) * spline_pred

print("Prediction function ready.")

In [ ]:
# Generate all-matchup predictions for men's
active_teams_m = teams_m[teams_m['LastD1Season'] >= CURRENT_SEASON]['TeamID'].tolist()
rows = []
for i, t1 in enumerate(active_teams_m):
    for t2 in active_teams_m[i+1:]:
        lo, hi = min(t1,t2), max(t1,t2)
        pred = predict_matchup(
            lo, hi, CURRENT_SEASON,
            model_m, win_pct_m, massey_feats_m, seeds_m,
            seed_spline, blend=1-BLEND_WEIGHT_MASSEY
        )
        rows.append({'ID': f"{CURRENT_SEASON}_{lo}_{hi}", 'Pred': np.clip(pred, 0.025, 0.975)})

sub_m = pd.DataFrame(rows)
print(f"Men's predictions: {len(sub_m)} matchups")

## Women's Model (no Massey — seed + win pct only)

In [ ]:
SKIP_WOMENS = False
sub_w = pd.DataFrame(columns=['ID', 'Pred'])  # default empty if women's data unavailable

try:
    reg_w = pd.read_csv(f"{DATA_DIR}/WRegularSeasonCompactResults.csv")
    tourney_w = pd.read_csv(f"{DATA_DIR}/WNCAATourneyCompactResults.csv")
    seeds_w = pd.read_csv(f"{DATA_DIR}/WNCAATourneySeeds.csv")
    teams_w = pd.read_csv(f"{DATA_DIR}/WTeams.csv")

    win_pct_w = compute_win_pcts(reg_w)

    train_tourney_w = tourney_w[(tourney_w['Season'] >= 2010) & (tourney_w['Season'] < CURRENT_SEASON)]
    features_w = build_matchup_features(train_tourney_w, win_pct_w, seeds_w, massey_df=None)

    feat_cols_w = [c for c in features_w.columns if c not in ['Season','Team1ID','Team2ID','Result']]
    X_w = features_w[feat_cols_w].fillna(0)
    y_w = features_w['Result']

    model_w = xgb.XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        use_label_encoder=False, eval_metric='logloss',
        random_state=42, verbosity=0,
    )
    model_w.fit(X_w, y_w)

    seed_diff_vals_w = features_w['SeedDiff'].values
    seed_spline_w = UnivariateSpline(
        sorted(seed_diff_vals_w),
        features_w.sort_values('SeedDiff')['Result'].values,
        s=len(seed_diff_vals_w), ext=3,
    )

    # Get active women's teams via men's cross-reference
    teams_m_df = pd.read_csv(f"{DATA_DIR}/MTeams.csv")
    active_names = set(teams_m_df[teams_m_df['LastD1Season'] >= CURRENT_SEASON]['TeamName'])
    active_teams_w = teams_w[teams_w['TeamName'].isin(active_names)]['TeamID'].tolist()

    rows_w = []
    for i, t1 in enumerate(active_teams_w):
        for t2 in active_teams_w[i+1:]:
            lo, hi = min(t1,t2), max(t1,t2)
            feat_row = {
                'SeedDiff': 0,
                'WPct1': 0.5,'WPct2': 0.5,'WPctH1':0.5,'WPctH2':0.5,
                'WPctA1':0.5,'WPctA2':0.5,'WPctN1':0.5,'WPctN2':0.5,
            }
            def gw_w(tid, col):
                r = win_pct_w[(win_pct_w['Season']==CURRENT_SEASON)&(win_pct_w['TeamID']==tid)]
                return r[col].values[0] if len(r) and col in r.columns else 0.5
            feat_row['WPct1'] = gw_w(lo,'Overall'); feat_row['WPct2'] = gw_w(hi,'Overall')
            feat_row['WPctH1'] = gw_w(lo,'H'); feat_row['WPctH2'] = gw_w(hi,'H')
            feat_row['WPctA1'] = gw_w(lo,'A'); feat_row['WPctA2'] = gw_w(hi,'A')
            feat_row['WPctN1'] = gw_w(lo,'N'); feat_row['WPctN2'] = gw_w(hi,'N')
            def gs_w(tid):
                r = seeds_w[(seeds_w['Season']==CURRENT_SEASON)&(seeds_w['TeamID']==tid)]
                if len(r): return int(r.iloc[0]['Seed'][1:3])
                return 8
            feat_row['SeedDiff'] = gs_w(lo) - gs_w(hi)

            row_df = pd.DataFrame([feat_row])[feat_cols_w].fillna(0)
            model_pred = float(model_w.predict_proba(row_df)[0, 1])
            # spline(SeedDiff): SeedDiff<0 means lo has better seed -> higher win prob
            spline_pred = float(np.clip(seed_spline_w(feat_row['SeedDiff']), 0.025, 0.975))
            pred = (1 - BLEND_WEIGHT_MASSEY) * model_pred + BLEND_WEIGHT_MASSEY * spline_pred
            rows_w.append({'ID': f"{CURRENT_SEASON}_{lo}_{hi}", 'Pred': np.clip(pred, 0.025, 0.975)})

    sub_w = pd.DataFrame(rows_w)
    print(f"Women's predictions: {len(sub_w)} matchups")

except (FileNotFoundError, Exception) as e:
    print(f"Women's data not found in {DATA_DIR} — skipping women's predictions")
    print(f"  Reason: {e}")
    print(f"  Tip: Use DATA_DIR = '../data/2026' for full data including women's.")
    SKIP_WOMENS = True

## Combine and Save

In [ ]:
combined = pd.concat([sub_m, sub_w], ignore_index=True)
out_dir = f"../output/{CURRENT_SEASON}/baseline_massey"
os.makedirs(out_dir, exist_ok=True)
out_path = f"{out_dir}/submission_combined.csv"
combined.to_csv(out_path, index=False)

print(f"Saved {len(combined)} rows to {out_path}")
print(f"  Men's:   {len(sub_m)}")
print(f"  Women's: {len(sub_w)}" + (" (SKIPPED — missing data)" if SKIP_WOMENS else ""))
combined.describe()